In [1]:
# Import required libraries

import os
import glob
import warnings

warnings.filterwarnings("ignore")

# Create required folders
folders = [
    "data/pdfs",
    "audio",
    "images",
    "db"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Project configuration
PROJECT_NAME = "Agri Voice"

LLM_MODEL = "llama3"
WHISPER_MODEL = "small"
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

# Default coordinates
LATITUDE = 20.90
LONGITUDE = 79.00

print("=" * 55)
print("🌾 AGRI VOICE")
print("=" * 55)

print("Project       :", PROJECT_NAME)
print("LLM           :", LLM_MODEL)
print("Whisper       :", WHISPER_MODEL)
print("Embedding     :", EMBEDDING_MODEL)
print("PDF folder    :", os.path.abspath("data/pdfs"))
print("=" * 55)
print("✅ Setup complete")

🌾 AGRI VOICE
Project       : Agri Voice
LLM           : llama3
Whisper       : small
Embedding     : intfloat/multilingual-e5-small
PDF folder    : e:\proj 2 Agri Voice\data\pdfs
✅ Setup complete


In [2]:
# FIX - INSTALL RAG LIBRARIES


%pip install chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

# FIX - INSTALL PDF + EMBEDDING LIBRARIES


%pip install pypdf sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

# TEST RAG LIBRARIES


import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print(" ChromaDB:", chromadb.__version__)
print(" PyPDF: OK")
print(" Sentence Transformers: OK")

 ChromaDB: 1.5.9
 PyPDF: OK
 Sentence Transformers: OK


In [5]:
## Now setting RAG libraries


import chromadb

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

print("Loading multilingual embedding model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("✅ ChromaDB imported")
print("✅ PDF reader imported")
print("✅ Embedding model loaded")

Loading multilingual embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ ChromaDB imported
✅ PDF reader imported
✅ Embedding model loaded


In [6]:
# Create a ChromaDB client
# ============================================================
# CELL 3
# ChromaDB Vector Database
# ============================================================

chroma_client = chromadb.PersistentClient(
    path="db"
)

collection = chroma_client.get_or_create_collection(
    name="agriculture_knowledge"
)

print("✅ ChromaDB database ready")
print("Collection:", collection.name)

✅ ChromaDB database ready
Collection: agriculture_knowledge


In [7]:
# ============================================================
# CELL 4
# Built-in Agricultural Knowledge
# ============================================================

agriculture_knowledge = [

    {
        "id": "cotton_pink_bollworm",
        "text": """
        Cotton pink bollworm is an important pest of cotton.
        Farmers should regularly monitor cotton fields for symptoms.
        Pheromone traps can be used for monitoring.
        Farmers should follow locally recommended integrated pest
        management practices and use registered pesticides only
        according to agricultural recommendations.
        """
    },

    {
        "id": "soybean_yellow_mosaic",
        "text": """
        Soybean yellow mosaic disease is associated with whitefly
        transmission. Farmers should monitor the crop regularly,
        remove severely affected plants where recommended, control
        the insect vector using locally recommended practices,
        and follow advice from agricultural authorities.
        """
    },

    {
        "id": "pigeon_pea_wilt",
        "text": """
        Pigeon pea wilt is a soil-borne disease.
        Management can include resistant varieties, healthy seed,
        proper field sanitation and recommended biological seed
        treatments such as Trichoderma where advised locally.
        """
    },

    {
        "id": "general_crop_care",
        "text": """
        Farmers should regularly inspect crops for insects, diseases,
        nutrient deficiency and water stress. Correct identification
        is important before applying pesticides. Farmers should follow
        local agricultural university, KVK or government recommendations.
        """
    },

    {
        "id": "irrigation",
        "text": """
        Irrigation should depend on crop stage, soil condition,
        rainfall and local weather. Avoid unnecessary irrigation
        when sufficient rainfall is expected.
        """
    }
]

print(
    f"✅ Loaded {len(agriculture_knowledge)} knowledge documents"
)

✅ Loaded 5 knowledge documents


In [8]:
# ============================================================
# CELL 5
# PDF Knowledge Ingestion
# ============================================================

pdf_files = glob.glob(
    "data/pdfs/*.pdf"
)

print(
    f"Found {len(pdf_files)} PDF file(s)"
)


def extract_pdf_text(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page in reader.pages:

        text = page.extract_text()

        if text:
            pages.append(text)

    return " ".join(pages)


pdf_documents = []

for pdf_path in pdf_files:

    try:

        text = extract_pdf_text(pdf_path)

        text = " ".join(
            text.split()
        )

        if text:

            # Create chunks
            chunk_size = 1200
            overlap = 200

            start = 0
            chunk_number = 0

            while start < len(text):

                end = start + chunk_size

                chunk = text[start:end]

                if chunk.strip():

                    pdf_documents.append(
                        {
                            "id": f"{os.path.basename(pdf_path)}_{chunk_number}",
                            "text": chunk
                        }
                    )

                start += chunk_size - overlap
                chunk_number += 1

        print(
            f"✅ Processed: {os.path.basename(pdf_path)}"
        )

    except Exception as error:

        print(
            f"❌ Error processing {pdf_path}: {error}"
        )


print(
    f"PDF chunks created: {len(pdf_documents)}"
)

Found 0 PDF file(s)
PDF chunks created: 0


In [9]:
# ============================================================
# CELL 6
# COMBINE AGRICULTURAL KNOWLEDGE
# ============================================================

all_documents = agriculture_knowledge + pdf_documents

print("=" * 50)
print("🌾 AGRI VOICE KNOWLEDGE BASE")
print("=" * 50)

print("Built-in documents :", len(agriculture_knowledge))
print("PDF chunks         :", len(pdf_documents))
print("Total documents    :", len(all_documents))

🌾 AGRI VOICE KNOWLEDGE BASE
Built-in documents : 5
PDF chunks         : 0
Total documents    : 5


In [10]:
# ============================================================
# CELL 7
# CREATE EMBEDDINGS + STORE IN CHROMADB
# ============================================================

document_ids = [item["id"] for item in all_documents]
document_texts = [item["text"] for item in all_documents]

print("🔄 Creating embeddings...")
print(f"Documents to embed: {len(document_texts)}")

document_embeddings = embedding_model.encode(
    ["passage: " + text for text in document_texts],
    normalize_embeddings=True,
    show_progress_bar=True
)

collection.upsert(
    ids=document_ids,
    documents=document_texts,
    embeddings=document_embeddings.tolist()
)

print()
print("=" * 50)
print(f"✅ Stored {len(document_texts)} documents in ChromaDB")
print("=" * 50)

🔄 Creating embeddings...
Documents to embed: 5


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Stored 5 documents in ChromaDB


In [11]:
# ============================================================
# CELL 8
# SEMANTIC SEARCH TEST
# ============================================================

def search_agriculture(question, top_k=4):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )

    documents = results.get("documents", [[]])[0]

    return documents


# Test question
test_question = "How can I control pink bollworm in cotton?"

results = search_agriculture(test_question, top_k=4)

print("=" * 60)
print("🔎 SEARCH QUESTION")
print("=" * 60)
print(test_question)

print("\n📚 RETRIEVED KNOWLEDGE")
print("=" * 60)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(result[:700])

🔎 SEARCH QUESTION
How can I control pink bollworm in cotton?

📚 RETRIEVED KNOWLEDGE

--- Result 1 ---

        Cotton pink bollworm is an important pest of cotton.
        Farmers should regularly monitor cotton fields for symptoms.
        Pheromone traps can be used for monitoring.
        Farmers should follow locally recommended integrated pest
        management practices and use registered pesticides only
        according to agricultural recommendations.
        

--- Result 2 ---

        Pigeon pea wilt is a soil-borne disease.
        Management can include resistant varieties, healthy seed,
        proper field sanitation and recommended biological seed
        treatments such as Trichoderma where advised locally.
        

--- Result 3 ---

        Soybean yellow mosaic disease is associated with whitefly
        transmission. Farmers should monitor the crop regularly,
        remove severely affected plants where recommended, control
        the insect vector using locally

In [12]:
# ============================================================
# CELL 9
# FASTER-WHISPER SPEECH-TO-TEXT
# ============================================================

from faster_whisper import WhisperModel

print("🔄 Loading Faster-Whisper model...")

whisper_model = WhisperModel(
    WHISPER_MODEL,
    device="cpu",
    compute_type="int8"
)

print("✅ Faster-Whisper loaded successfully")
print("Model:", WHISPER_MODEL)
print("Device: CPU")
print("Compute type: INT8")

🔄 Loading Faster-Whisper model...
✅ Faster-Whisper loaded successfully
Model: small
Device: CPU
Compute type: INT8


In [13]:

# FIX - INSTALL FASTER-WHISPER


%pip install faster-whisper

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# importing required libraries for audio recording and processing
%pip install sounddevice scipy requests ollama gtts transformers torch torchvision pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:

# CELL 10
# MICROPHONE AUDIO RECORDING


import sounddevice as sd
from scipy.io.wavfile import write

def record_audio(
    duration=6,
    filename="audio/question.wav",
    sample_rate=16000
):
    print(f"🎤 Speak now for {duration} seconds...")

    recording = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="int16"
    )

    sd.wait()

    write(
        filename,
        sample_rate,
        recording
    )

    print("✅ Audio saved:", filename)

    return filename

print("✅ Microphone recording function ready")

✅ Microphone recording function ready


In [16]:
import sounddevice
import scipy
import requests
import ollama
import gtts
import transformers
import torch
import torchvision
import PIL

print("✅ sounddevice")
print("✅ scipy")
print("✅ requests")
print("✅ ollama")
print("✅ gTTS")
print("✅ transformers")
print("✅ torch")
print("✅ torchvision")
print("✅ pillow")

✅ sounddevice
✅ scipy
✅ requests
✅ ollama
✅ gTTS
✅ transformers
✅ torch
✅ torchvision
✅ pillow


In [30]:
# CELL 11 
# SPEECH-TO-TEXT CONVERSION
# IMPROVED SPEECH-TO-TEXT

import numpy as np
from scipy.io.wavfile import read


AGRICULTURE_PROMPT = """
Agriculture, farming, farmer, cotton, soybean, pigeon pea,
red chili, pink bollworm, pest, disease, crop, irrigation,
rainfall, fertilizer, soil, leaf, insect, pesticide, weather.
"""


def transcribe_audio(audio_path):

    # Read WAV file directly
    sample_rate, audio = read(audio_path)

    # Convert to float32
    audio = audio.astype(np.float32)

    # Normalize int16 audio
    if np.max(np.abs(audio)) > 1:
        audio = audio / 32768.0

    # Convert stereo to mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    # Transcribe
    segments, information = whisper_model.transcribe(
        audio,
        beam_size=5,
        initial_prompt=AGRICULTURE_PROMPT
    )

    text = " ".join(
        segment.text.strip()
        for segment in segments
    ).strip()

    language = information.language

    return text, language


print("✅ Improved speech-to-text function ready")

✅ Improved speech-to-text function ready


In [31]:
# FIX - FASTER-WHISPER / PYAV COMPATIBILITY
%pip install --upgrade av faster-whisper

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
# faster whisper model initialization
from faster_whisper import WhisperModel

WHISPER_MODEL = "base"

print("🔄 Loading Faster-Whisper...")
print("Model:", WHISPER_MODEL)
print("Device: CPU")
print("Compute type: INT8")

whisper_model = WhisperModel(
    WHISPER_MODEL,
    device="cpu",
    compute_type="int8",
    cpu_threads=2,
    num_workers=1
)

print("✅ Faster-Whisper is ready")

🔄 Loading Faster-Whisper...
Model: base
Device: CPU
Compute type: INT8
✅ Faster-Whisper is ready


In [37]:
import av
import faster_whisper

print("PyAV version:", av.__version__)
print("Faster-Whisper version:", faster_whisper.__version__)

PyAV version: 19.0.1
Faster-Whisper version: 1.2.1


In [38]:

# CELL 12
# TEST VOICE INPUT

audio_path = record_audio(
    duration=6,
    filename="audio/question.wav"
)

question, language = transcribe_audio(audio_path)

print("\n" + "=" * 55)
print("🎤 FARMER'S QUESTION")
print("=" * 55)

print("Question :", question)
print("Language :", language)

print("=" * 55)

🎤 Speak now for 6 seconds...
✅ Audio saved: audio/question.wav

🎤 FARMER'S QUESTION
Question : How can I control bollworm in cotton?
Language : en


In [39]:
# CELL 13 — WEATHER SERVICE

import requests

def get_weather():
    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": LATITUDE,
                "longitude": LONGITUDE,
                "current": (
                    "temperature_2m,"
                    "relative_humidity_2m,"
                    "precipitation"
                ),
                "timezone": "auto"
            },
            timeout=15
        )

        response.raise_for_status()

        data = response.json()
        current = data["current"]

        temperature = current["temperature_2m"]
        humidity = current["relative_humidity_2m"]
        precipitation = current["precipitation"]

        return (
            f"Current temperature: {temperature}°C\n"
            f"Humidity: {humidity}%\n"
            f"Precipitation: {precipitation} mm"
        )

    except Exception as error:
        return f"Weather service unavailable: {error}"


print("🌦️ Testing Weather Service...")
print("=" * 50)
print(get_weather())

🌦️ Testing Weather Service...
Current temperature: 27.7°C
Humidity: 58%
Precipitation: 0.0 mm


In [40]:

# CELL 14 — QUERY ROUTER

WEATHER_KEYWORDS = [
    "weather",
    "rain",
    "rainfall",
    "temperature",
    "humidity",
    "forecast",
    "climate",
    "बारिश",
    "वर्षा",
    "मौसम",
    "तापमान",
    "पाऊस",
    "हवामान"
]


def route_query(question):
    """
    Decide whether the farmer's question
    is related to weather or agriculture.
    """

    question_lower = question.lower()

    for keyword in WEATHER_KEYWORDS:
        if keyword in question_lower:
            return "weather"

    return "agriculture"


print("✅ Query router is ready")

✅ Query router is ready


In [41]:
## Testing the query router

test_questions = [
    "What is the weather today?",
    "Will there be rain tomorrow?",
    "What is the temperature today?",
    "How can I control pink bollworm in cotton?",
    "Why are my soybean leaves turning yellow?",
    "How should I manage pigeon pea wilt?"
]


print("=" * 60)
print("🧠 AGRI VOICE QUERY ROUTER TEST")
print("=" * 60)

for question in test_questions:

    intent = route_query(question)

    print("\nQuestion :", question)
    print("Intent   :", intent)
    print("-" * 60)

🧠 AGRI VOICE QUERY ROUTER TEST

Question : What is the weather today?
Intent   : weather
------------------------------------------------------------

Question : Will there be rain tomorrow?
Intent   : weather
------------------------------------------------------------

Question : What is the temperature today?
Intent   : weather
------------------------------------------------------------

Question : How can I control pink bollworm in cotton?
Intent   : agriculture
------------------------------------------------------------

Question : Why are my soybean leaves turning yellow?
Intent   : agriculture
------------------------------------------------------------

Question : How should I manage pigeon pea wilt?
Intent   : agriculture
------------------------------------------------------------
